Redes Neuronales para Lenguaje Natural, 2024

---
# **Análisis de sentimento con BERT en español (BETO)**

En este notebook haremos experimentos utilizando modelos de tipo BERT:
- Tokenización de oraciones.
- Fine-tuning de un analizador de sentimiento. Para esto utilizaremos un corpus de comentarios sobre preoductos, películas y restaurantes traducidos al español.
- Experimentos sobre uso de modelo de lenguaje enmascarado.

---



Empezaremos por instalar la librería transformers.

In [22]:
!pip install transformers

Algunos imports que utilizaremos a lo largo de este notebook y algunas definiciones:

In [23]:
import time

import pandas as pd
import torch

import transformers
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification
from transformers import DistilBertForMaskedLM

torch.backends.cudnn.deterministic = True
RANDOM_SEED = 248
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(DEVICE)

cuda


## **Dataset**

Vamos a utilizar una versión traducida al español del dataset presentado en *'From Group to Individual Labels using Deep Features'*, Kotzias et al. (2015), que es una compilación de reviews sobre productos, películas y restaurantes. Cada review tiene una valoración de 0 (negativo) o 1 (positivo). Ejecute el siguiente bloque para descargarlo:

In [ ]:
! wget https://www.fing.edu.uy/owncloud/index.php/s/DRm5GJnSBbL6rny/download/senti_corpus.zip
! unzip senti_corpus.zip

--2024-10-14 22:38:57--  https://www.fing.edu.uy/owncloud/index.php/s/DRm5GJnSBbL6rny/download/senti_corpus.zip
Resolving www.fing.edu.uy (www.fing.edu.uy)... 164.73.32.20
Connecting to www.fing.edu.uy (www.fing.edu.uy)|164.73.32.20|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 89239 (87K) [application/zip]
Saving to: ‘senti_corpus.zip.1’

senti_corpus.zip.1  100%[===================>]  87.15K   187KB/s    in 0.5s    

2024-10-14 22:38:59 (187 KB/s) - ‘senti_corpus.zip.1’ saved [89239/89239]

Archive:  senti_corpus.zip
replace senti-dev.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

Cargamos el archivo en memoria con pandas:

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

train_df = pd.read_csv('./senti-train.tsv',sep='\\t')
dev_df = pd.read_csv('./senti-dev.tsv',sep='\\t')
test_df = pd.read_csv('./senti-test.tsv',sep='\\t')

train_df.head()

Desplegamos la cantidad de filas y columnas de nuestros datos, y los separamos en textos y labels:

In [ ]:
print("Train:",train_df.shape)
print("Dev:",dev_df.shape)
print("Test:",test_df.shape)

train_texts = train_df.iloc[:,0].values
train_labels = train_df.iloc[:,1].values

dev_texts = dev_df.iloc[:,0].values
dev_labels = dev_df.iloc[:,1].values

test_texts = test_df.iloc[:,0].values
test_labels = test_df.iloc[:,1].values

print(train_texts[0])
print(train_labels[0])

## **Tokenizamos**

Para utilizar un modelo preentrenado BERT es preciso tokenizar la entrada de la misma manera con la que el modelo fue preentrenado. En nuestro caso utilizaremos la versión de BERT en español **BETO** creada por la Universidad de Chile, pero en su variante DistilBERT que es una versión más pequeña y más rápida (https://huggingface.co/docs/transformers/model_doc/distilbert). En particular utilizaremos "distilbert-base-spanish-uncased", por lo tanto cargamos el tokenizer correspondiente:

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('dccuchile/distilbert-base-spanish-uncased')

print("Vocabulario:",tokenizer.vocab_size)
print("Índice de 'perro':",tokenizer.vocab["perro"])
print("Índice de [MASK]:",tokenizer.vocab["[MASK]"])
print("Índice de [CLS]:",tokenizer.vocab["[CLS]"])
print("Índice de [SEP]:",tokenizer.vocab["[SEP]"])

Observemos cómo funciona el tokenizador con un par de ejemplos.
Utilice el método del tokenizador para obtener la representación de una oración y de un par de oraciones.
- ¿Qué clase se utiliza para almacenar una representación tokenizada?
- ¿Qué atributos tiene esta representación? Observe qué devuelve al imprimir el contenido de la variable y al imprimir su atributo "encodings".

In [ ]:
sentence1 = "Juan habla rapidísimo y lleno de uruguayismos."
sentence2 = "Uruguay es el mejor país."

# sentence_encoding = # su código aquí
# pair_encoding = # su código aquí
sentence_encoding = tokenizer([sentence1, sentence2])

print(type(sentence_encoding))

print(sentence_encoding)
print(sentence_encoding[0])

print(sentence_encoding.encodings[0].ids)
print(sentence_encoding.encodings[0].type_ids)
print(sentence_encoding.encodings[0].tokens)
print(sentence_encoding.encodings[0].offsets)
print(sentence_encoding.encodings[0].attention_mask)
print(sentence_encoding.encodings[0].special_tokens_mask)
print(sentence_encoding.encodings[0].overflowing)

print(sentence_encoding.encodings[1].ids)
print(sentence_encoding.encodings[1].type_ids)
print(sentence_encoding.encodings[1].tokens)
print(sentence_encoding.encodings[1].offsets)
print(sentence_encoding.encodings[1].attention_mask)
print(sentence_encoding.encodings[1].special_tokens_mask)
print(sentence_encoding.encodings[1].overflowing)

pair_encoding = tokenizer(sentence1, sentence2)
print(pair_encoding)
print(pair_encoding.encodings)

print(pair_encoding.encodings[0].ids)
print(pair_encoding.encodings[0].type_ids)
print(pair_encoding.encodings[0].tokens)
print(pair_encoding.encodings[0].offsets)
print(pair_encoding.encodings[0].attention_mask)
print(pair_encoding.encodings[0].special_tokens_mask)
print(pair_encoding.encodings[0].overflowing)


Ahora tokenice las tres particiones de train, dev y test. Para eso utilice los argumentos truncation=True y padding=True.

In [ ]:
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
dev_encodings = tokenizer(list(dev_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

Analicemos un elemento tokenizado:

In [ ]:
print(train_encodings[0])
print(train_encodings[0].tokens)
print(train_encodings[0].tokens.index('[PAD]'))
print(train_encodings[0].attention_mask.index(0))
print(len(train_encodings[0].tokens))

Observe algunos ejemplos y analice el contenido de cada campo (por dudas consulte https://huggingface.co/docs/tokenizers/api/encoding).

Responda:
- ¿Qué objetivo cumple el campo attention_mask?
- ¿Qué objetivo cumple el campo special_tokens_mask?

In [ ]:
print(train_encodings[0].attention_mask)
print(train_encodings[0].special_tokens_mask)



Una vez tokenizados, instanciamos los datos como Dataset de pytorch y los metemos en un DataLoader para cada partición.

In [ ]:
class SentiCorpusDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentiCorpusDataset(train_encodings, train_labels)
dev_dataset = SentiCorpusDataset(dev_encodings, dev_labels)
test_dataset = SentiCorpusDataset(test_encodings, test_labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = torch.utils.data.DataLoader(dev_dataset, batch_size=16, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)


## **Carga y *fine-tuning* de un modelo BERT preentrenado**

Cargamos "distilbert-base-spanish-uncased":

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('dccuchile/distilbert-base-spanish-uncased')
model.to(DEVICE)

Evaluaremos los modelos resultantes con la siguente función para calcular el accuracy.

Notar que al invocar el forward del modelo, se deben enviar los inputs (tokens) y además la máscada de atención obtenida como resultado de la tokenización.

In [ ]:
def compute_accuracy(model, data_loader, device):
    with torch.no_grad():
        correct_pred = 0
        num_examples = 0

        for batch_idx, batch in enumerate(data_loader):

            ### Obtener los token ids y la máscara de atención
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            ### Pasada forward
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs['logits']
            predicted_labels = torch.argmax(logits, 1)

            num_examples += labels.size(0)
            correct_pred += (predicted_labels == labels).sum()

        return correct_pred.float()/num_examples * 100

In [ ]:
compute_accuracy(model,test_loader,DEVICE)

Completar el siguiente ciclo de entrenamiento para realizar el fine-tuning del modelo preentrenado. En este caso además de enviar los tokens y la máscara de atención, se le deben enviar al modelo los labels esperados, y como parte de las salidas se obtendrá el *loss*.

In [ ]:
NUM_EPOCHS = 5
optim = torch.optim.Adam(model.parameters(), lr=5e-6)

start_time = time.time()

for epoch in range(NUM_EPOCHS):

    model.train()

    for batch_idx, batch in enumerate(train_loader):

        ### Obtener los token ids, la máscara de atención y los labels
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        ### Pasada forward, invocar el modelo y obtener:
        ### - el loss para optimizar
        ### - los logits para calcular la performance
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss, logits = outputs['loss'], outputs['logits']

        ### Pasada backward
        optim.zero_grad()
        loss.backward()
        optim.step()

        ### Logging
        if not batch_idx % 20:
            print (f'Epoch: {epoch+1:04d}/{NUM_EPOCHS:04d} | '
                   f'Batch {batch_idx:04d}/{len(train_loader):04d} | '
                   f'Loss: {loss:.4f}')

    model.eval()

    with torch.set_grad_enabled(False):
        print(f'Accuracy en training: '
              f'{compute_accuracy(model, train_loader, DEVICE):.2f}%'
              f'\nAccuracy en dev: '
              f'{compute_accuracy(model, dev_loader, DEVICE):.2f}%')

    print(f'Tiempo: {(time.time() - start_time)/60:.2f} min')

print(f'Tiempo total: {(time.time() - start_time)/60:.2f} min')


Para completar el experimento evaluamos en **test** el modelo resultante del fine-tuning:

In [ ]:
print(f'Accuracy en test: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

# **(opcional) Uso del MLM**

Ahora veremos el uso de BERT preentrenado como modelo de lenguaje enmascarado. Lo utilizaremos para predecir algunas palabras posibles para completar de la siguiente manera:

- Instancie un modelo de BERT preentrenado para MLM usando la clase DistilBertForMaskedLM
- Tokenize algunas oraciones, asegurándose de que incluyan al menos un token [MASK]
- Haga la pasada forward de la red y obtenga los K tokens más probables predichos para cada máscara. ¿Las palabras predichas por el modelo se corresponden con su intuición?

In [ ]:
# Ejemplos de oraciones (puede agregar todas las que quiera para probar)
sentence1 = "[MASK] es el mejor país."
sentence2 = "Caniche es una raza de [MASK]."
sentence3 = "Juan [MASK] una carrera."


In [ ]:
# Cargar el modelo tipo MLM
# mlm = # Escriba su código a partir de aquí

mlm = DistilBertForMaskedLM.from_pretrained('dccuchile/distilbert-base-spanish-uncased')
mlm.to(DEVICE)


In [ ]:
def get_mlm_result(sentence, model, k):
    # Este método obtiene una oración con un token [MASK], un modelo y una cantidad de palabras k
    # y devuelve las k palabras más probables predichas por el modelo en esa posición
    # Escriba su código a partir de aquí
    pass

def get_mlm_result(sentence, model, k):
    with torch.no_grad():
        sentence_encoding = tokenizer(sentence, padding=True)
        mask_idx = sentence_encoding.encodings[0].ids.index(tokenizer.mask_token_id)

        input_ids = torch.tensor([sentence_encoding.encodings[0].ids]).to(DEVICE)
        attention_mask = torch.tensor(sentence_encoding.encodings[0].attention_mask).to(DEVICE)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs['logits']

        outputs = mlm(input_ids, attention_mask=attention_mask)
        logits = outputs['logits']
        maxlogit = torch.topk(logits[0][mask_idx],k=k).indices
        return tokenizer.convert_ids_to_tokens(list(maxlogit))

print(sentence1, get_mlm_result(sentence1, mlm, 3))
print(sentence2, get_mlm_result(sentence2, mlm, 3))
print(sentence3, get_mlm_result(sentence3, mlm, 3))
